# Bulk C–R axis test, Gemma 3 12B: composed map or own-step shortcut? (preregistered: `docs/preregistration_axis_bulk.md`)

5,250 new "which is farther east / north, C or R?" questions, scored clean, with the map erased and with the fair random
control. Reuses two files from earlier runs that are already on your Drive in `MyDrive/geomap/results/erase/`:
`gemma3-12b_erase_axis_state.pt` (the map erasure) and `gemma3-12b_erase_fair_control.pt` (the fair control).

**Before you press Run all:**
1. Upload the new `geomap_colab.zip` to Google Drive at **`MyDrive/geomap/geomap_colab.zip`** (replace the old one).
2. Colab secrets (key icon), notebook access on: **`HF_TOKEN`** (required), `SLACK_WEBHOOK_URL` (optional).
3. **Runtime → Change runtime type → A100** (G4 or H100 work too).

Then **Runtime → Run all**. Expected time: about 30–45 minutes (model load plus three scoring passes). If Colab
disconnects, press **Run all** again; finished scoring passes are skipped.


In [ ]:
# 1. GPU check
import subprocess, torch, os
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100"
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
assert gb > 35, f"Only {gb:.0f} GB of GPU memory: pick an A100 (40 GB)"


In [ ]:
# 2. Code from Drive; results live ON Drive (saved as they are produced, kept across disconnects)
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, zipfile
D = "/content/drive/MyDrive/geomap"
BUNDLE, RES = f"{D}/geomap_colab.zip", f"{D}/results"
assert os.path.exists(BUNDLE), f"Upload geomap_colab.zip to {BUNDLE}"
shutil.rmtree("/content/geomap", ignore_errors=True)
zipfile.ZipFile(BUNDLE).extractall("/content/geomap")
if not os.path.exists(RES):
    shutil.copytree("/content/geomap/results", RES)
assert os.path.exists("/content/geomap/scripts/erase_axis_bulk.py"), "Old bundle: upload the new geomap_colab.zip"
# The two erasures from earlier runs must be on Drive (they are not in the bundle)
for f in ["gemma3-12b_erase_axis_state.pt", "gemma3-12b_erase_fair_control.pt"]:
    p = f"{RES}/erase/{f}"
    assert os.path.exists(p) and os.path.getsize(p) > 10000, f"missing on Drive: {p} (from the earlier erasure runs)"
shutil.rmtree("/content/geomap/results")
os.symlink(RES, "/content/geomap/results")
os.chdir("/content/geomap")
print("already saved:", sorted(os.listdir(f"{RES}/erase")) if os.path.exists(f"{RES}/erase") else "nothing yet")

In [ ]:
# 3. Dependencies, PyTorch only (Colab's TensorFlow breaks transformers' Gemma 3 import)
!pip -q install "transformers==4.57.3" nnsight accelerate
import sys
os.environ.update(PYTHONPATH="/content/geomap/src", USE_TF="0", USE_FLAX="0", USE_TORCH="1",
                  TRANSFORMERS_NO_TF="1", TF_CPP_MIN_LOG_LEVEL="3")
sys.path.insert(0, "/content/geomap/src")
!python -c "from transformers.models.gemma3.modeling_gemma3 import Gemma3ForConditionalGeneration; print('Gemma 3 import OK')"

In [ ]:
# 4. Secrets: Hugging Face (required) and Slack (optional)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
from huggingface_hub import whoami
print("Hugging Face:", whoami(token=os.environ["HF_TOKEN"])["name"])
try:
    os.environ["SLACK_WEBHOOK_URL"] = userdata.get("SLACK_WEBHOOK_URL")
except Exception:
    pass
import sys
sys.modules.pop("geomap", None)  # a stale or wrong "geomap" import must not shadow the code
sys.path.insert(0, "/content/geomap/src")
from geomap.checkpoint import notify
notify("[Axis bulk 12B] Colab run started (or resumed)")

In [ ]:
# 5. Bulk C–R axis test: three scoring passes (resumes if interrupted)
!cd /content/geomap && PYTHONPATH=/content/geomap/src USE_TF=0 USE_FLAX=0 USE_TORCH=1 TRANSFORMERS_NO_TF=1 python scripts/erase_axis_bulk.py --model gemma3-12b 2>&1 | grep -v -i warning


In [ ]:
# 6. One zip of the results + the results log, on Drive and downloaded here
import glob
with zipfile.ZipFile("/content/axis_bulk_12b_results.zip", "w") as z:
    for f in glob.glob(f"{RES}/erase/gemma3-12b_axis_bulk*"):
        z.write(f, "results/erase/" + os.path.basename(f))
    z.write(f"{RES}/RESULTS.md", "results/RESULTS.md")
shutil.copy("/content/axis_bulk_12b_results.zip", f"{D}/axis_bulk_12b_results.zip")
notify("[Axis bulk 12B] all done: results in MyDrive/geomap/axis_bulk_12b_results.zip")
from google.colab import files
files.download("/content/axis_bulk_12b_results.zip")
